# DEPICT Analysis 1A — all association panels as standalone grand-figure components

This notebook exports every Analysis 1A association panel as an independent,
fully vectorized PDF using the same dimensions, typography, point styling,
fold-specific fitted lines, pooled Pearson line and annotation used for panels
A–D in `DEPICT_GrandFigure_IndividualComponents_Only_v2_Pearson.ipynb`.

It produces:

- 4 baseline-state similarity panels;
- 4 mean condition-wise response-similarity panels;
- 4 global response-landscape similarity panels;
- 4 drug training-exposure panels;
- 4 drug–dose–duration training-exposure panels;
- one standalone fold-color legend;
- standalone x-axis title components for each similarity/exposure definition;
- a CSV manifest and the exact data used for every panel.

Panel letters are intentionally omitted from the individual PDFs so they can be
assigned during final figure assembly.

In [ ]:
from __future__ import annotations

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.stats import pearsonr

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

## 1. Paths and unified publication style

In [ ]:
PROJECT_ROOT = Path("~/DEPICT")
MAIN_DIR = PROJECT_ROOT / "Code/downstream_analysis_code/TransferabilityUnseenCell"
OUT_DIR = (
    MAIN_DIR
    / "DEPICT_Analysis1A_AllAssociation_IndividualComponents_Pearson"
)
TABLE_OUT_DIR = OUT_DIR / "plot_data"
OUT_DIR.mkdir(parents=True, exist_ok=True)
TABLE_OUT_DIR.mkdir(parents=True, exist_ok=True)

PATHS = {
    "baseline": (
        MAIN_DIR / "a1_ood_similarity"
        / "tables" / "a1_primary_cell_level_metrics.csv"
    ),
    "response_mean_condition": (
        MAIN_DIR
        / "a1_response_similarity_mean_condition_v3" / "tables"
        / "response_similarity_cell_level_analysis.csv"
    ),
    "response_global_landscape": (
        MAIN_DIR
        / "a1_response_similarity_global_landscape_v3" / "tables"
        / "response_similarity_cell_level_analysis.csv"
    ),
    "drug_exposure": (
        MAIN_DIR
        / "a1_drug_training_exposure" / "tables"
        / "training_exposure_cell_level_analysis.csv"
    ),
    "condition_exposure": (
        MAIN_DIR
        / "a1_condition_training_exposure" / "tables"
        / "training_exposure_cell_level_analysis.csv"
    ),
}

# If optimized v3 analyses were written to different folders, change only the
# two exposure paths above. The required column names remain unchanged.

FOLD_ORDER = [f"cell_split{i}" for i in range(1, 6)]
FOLD_COLORS = {
    "cell_split1": "#0072B2",
    "cell_split2": "#E69F00",
    "cell_split3": "#009E73",
    "cell_split4": "#CC79A7",
    "cell_split5": "#D55E00",
}

# Exact panel size and style inherited from the grand-figure component notebook.
SIZE_PANEL = (3.35, 3.0)

mpl.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 9.0,
    "axes.labelsize": 9.5,
    "axes.titlesize": 10.5,
    "xtick.labelsize": 8.3,
    "ytick.labelsize": 8.3,
    "legend.fontsize": 8.0,
    "axes.linewidth": 0.85,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})

print("Output directory:", OUT_DIR)
for name, path in PATHS.items():
    print(f"{name}: {path}")

## 2. Load and audit all five Analysis 1A cell-level tables

In [ ]:
OUTCOMES = [
    ("delta_pcc", "Mean ΔPCC within held-out cell"),
    ("delta_r2", "Mean ΔR² within held-out cell"),
    ("dge_mse", "Mean DGE MSE within held-out cell"),
    ("mse_gain_over_naive", "Mean DGE-MSE gain over naive baseline"),
]

def require(condition, message):
    if not condition:
        raise RuntimeError(message)

def read_table(path: Path) -> pd.DataFrame:
    path = Path(path)
    if path.exists():
        return pd.read_csv(path)

    parquet_path = path.with_suffix(".parquet")
    if parquet_path.exists():
        return pd.read_parquet(parquet_path)

    raise FileNotFoundError(
        f"Could not find CSV or Parquet table for: {path}"
    )

def audit_table(df, x_col, label, require_eligibility=False):
    required = {
        "split_type",
        "cell_id",
        x_col,
        *(name for name, _ in OUTCOMES),
    }
    missing = required.difference(df.columns)
    require(
        not missing,
        f"{label} table is missing columns: {sorted(missing)}",
    )

    df = df.copy()
    df["split_type"] = df["split_type"].astype(str)
    df["cell_id"] = df["cell_id"].astype(str)

    if require_eligibility and "eligible_response_similarity" in df.columns:
        raw = df["eligible_response_similarity"]
        if raw.dtype == bool:
            eligible = raw
        else:
            eligible = raw.astype(str).str.lower().isin(
                {"true", "1", "yes"}
            )
        df = df.loc[eligible].copy()

    require(not df.empty, f"{label} table has no eligible rows.")
    require(
        not df.duplicated(["split_type", "cell_id"]).any(),
        f"{label} has duplicate split × cell rows.",
    )

    unexpected_folds = set(df["split_type"]).difference(FOLD_ORDER)
    require(
        not unexpected_folds,
        f"{label} has unexpected split labels: {sorted(unexpected_folds)}",
    )

    return df

baseline = audit_table(
    read_table(PATHS["baseline"]),
    "nearest_training_similarity_pearson",
    "Baseline-state similarity",
)

response_mean = audit_table(
    read_table(PATHS["response_mean_condition"]),
    "nearest_training_cell_response",
    "Mean condition-wise response similarity",
    require_eligibility=True,
)

response_global = audit_table(
    read_table(PATHS["response_global_landscape"]),
    "nearest_training_cell_response",
    "Global response-landscape similarity",
    require_eligibility=True,
)

drug_exposure = audit_table(
    read_table(PATHS["drug_exposure"]),
    "drug_training_exposure_fraction",
    "Drug training exposure",
)

condition_exposure = audit_table(
    read_table(PATHS["condition_exposure"]),
    "condition_training_exposure_fraction",
    "Drug-dose-duration training exposure",
)

print("Rows loaded:")
print("  baseline:", len(baseline))
print("  mean condition-wise response:", len(response_mean))
print("  global response landscape:", len(response_global))
print("  drug exposure:", len(drug_exposure))
print("  condition exposure:", len(condition_exposure))

## 3. Unified standalone-component plotting helpers

In [ ]:
def save_pdf(fig, filename, dpi=None, pad=0.04):
    path = OUT_DIR / filename
    kwargs = {
        "format": "pdf",
        "bbox_inches": "tight",
        "pad_inches": pad,
    }
    if dpi is not None:
        kwargs["dpi"] = dpi
    fig.savefig(path, **kwargs)
    plt.close(fig)
    print("Saved:", path)

def save_standalone_legend(
    handles,
    labels,
    filename,
    title=None,
    ncol=1,
    figsize=(5.2, 1.4),
    fontsize=8.0,
    title_fontsize=8.5,
):
    fig, ax = plt.subplots(figsize=figsize)
    ax.axis("off")
    legend = ax.legend(
        handles,
        labels,
        loc="center",
        frameon=False,
        title=title,
        ncol=ncol,
        handletextpad=0.55,
        columnspacing=1.0,
        labelspacing=0.42,
        borderaxespad=0,
        fontsize=fontsize,
    )
    if legend.get_title() is not None:
        legend.get_title().set_fontsize(title_fontsize)
        legend.get_title().set_fontweight("bold")
    save_pdf(fig, filename, pad=0.02)

def save_standalone_axis_title(text, filename, figsize=(4.9, 0.42)):
    fig, ax = plt.subplots(figsize=figsize)
    ax.axis("off")
    ax.text(
        0.5,
        0.5,
        text,
        ha="center",
        va="center",
        fontsize=9.5,
    )
    save_pdf(fig, filename, pad=0.01)

def safe_pearson(x, y):
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    valid = np.isfinite(x) & np.isfinite(y)
    x, y = x[valid], y[valid]
    if (
        len(x) < 3
        or np.unique(x).size < 2
        or np.unique(y).size < 2
    ):
        return np.nan
    return float(pearsonr(x, y).statistic)

def safe_ols(x, y):
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    valid = np.isfinite(x) & np.isfinite(y)
    x, y = x[valid], y[valid]
    if len(x) < 3 or np.unique(x).size < 2:
        return np.nan, np.nan
    return tuple(np.polyfit(x, y, 1))

def padded_limits(values, pad_fraction=0.08, min_pad=0.01):
    values = np.asarray(values, float)
    values = values[np.isfinite(values)]
    require(len(values) > 0, "Cannot determine limits from empty values.")
    lo, hi = values.min(), values.max()
    pad = max((hi - lo) * pad_fraction, min_pad)
    return lo - pad, hi + pad

def finite_xy(df, x_col, y_col):
    out = (
        df[["split_type", "cell_id", x_col, y_col]]
        .copy()
        .rename(columns={x_col: "x", y_col: "y"})
    )
    return out.loc[
        np.isfinite(out["x"]) & np.isfinite(out["y"])
    ].copy()

def plot_single_association_component(
    *,
    df,
    analysis_id,
    x_col,
    y_col,
    y_label,
    selected_labels,
    filename,
):
    """
    Plot one standalone panel using the exact visual conventions used by the
    baseline A–D grand-figure component notebook.

    The x-axis title is intentionally omitted and exported separately.
    """
    p = finite_xy(df, x_col, y_col)
    require(not p.empty, f"{analysis_id}/{y_col}: no finite observations.")

    xlim = padded_limits(p["x"], 0.06, 0.005)
    ylim = padded_limits(p["y"], 0.08, 0.01)

    fig, ax = plt.subplots(figsize=SIZE_PANEL)

    for fold in FOLD_ORDER:
        sub = p.loc[p["split_type"].astype(str) == fold]
        if sub.empty:
            continue

        color = FOLD_COLORS[fold]
        ax.scatter(
            sub["x"],
            sub["y"],
            s=25,
            color=color,
            alpha=0.90,
            edgecolor="white",
            linewidth=0.4,
            zorder=3,
        )

        slope, intercept = safe_ols(sub["x"], sub["y"])
        if np.isfinite(slope):
            xs = np.linspace(*xlim, 150)
            ax.plot(
                xs,
                slope * xs + intercept,
                color=color,
                linewidth=1.45,
                alpha=0.85,
                zorder=2,
            )

    pooled_slope, pooled_intercept = safe_ols(p["x"], p["y"])
    pooled_r = safe_pearson(p["x"], p["y"])

    if np.isfinite(pooled_slope):
        xs = np.linspace(*xlim, 200)
        pooled_line, = ax.plot(
            xs,
            pooled_slope * xs + pooled_intercept,
            color="black",
            linestyle="--",
            linewidth=1.75,
            zorder=4,
        )
        ax.legend(
            [pooled_line],
            [f"Pooled r = {pooled_r:.2f}"],
            loc="lower right",
            frameon=False,
            fontsize=7.5,
            handlelength=2.0,
            borderaxespad=0.25,
        )

    for _, row in p.loc[
        p["cell_id"].astype(str).isin(selected_labels)
    ].iterrows():
        ax.annotate(
            str(row["cell_id"]),
            (row["x"], row["y"]),
            xytext=(3, 3),
            textcoords="offset points",
            fontsize=5.8,
            color="0.15",
        )

    ax.grid(
        axis="y",
        color="0.2",
        alpha=0.13,
        linewidth=0.6,
    )
    ax.set_axisbelow(True)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_xlabel("")
    ax.set_ylabel(y_label)

    fig.tight_layout(pad=0.45)
    save_pdf(fig, filename)

    export = p.copy()
    export["analysis_id"] = analysis_id
    export["x_source_column"] = x_col
    export["outcome"] = y_col
    export["pooled_pearson_r"] = pooled_r
    export["pooled_ols_slope"] = pooled_slope
    export["pooled_ols_intercept"] = pooled_intercept
    return export

## 4. Define the five analysis groups and their standalone components

In [ ]:
ANALYSIS_GROUPS = [
    {
        "analysis_id": "baseline_state_similarity",
        "file_prefix": "Baseline",
        "df": baseline,
        "x_col": "nearest_training_similarity_pearson",
        "x_title": "Nearest training-cell similarity (Pearson correlation)",
        "labels": {
            "delta_pcc": ["U937", "SKM1", "U266", "HT115"],
            "delta_r2": ["U937", "SKM1", "U266"],
            "dge_mse": ["U937", "SKM1", "U266"],
            "mse_gain_over_naive": ["U937", "SKM1", "U266"],
        },
    },
    {
        "analysis_id": "mean_conditionwise_response_similarity",
        "file_prefix": "ResponseMean",
        "df": response_mean,
        "x_col": "nearest_training_cell_response",
        "x_title": "Mean condition-wise DGE Pearson correlation",
        "labels": {name: ["SKM1", "U937"] for name, _ in OUTCOMES},
    },
    {
        "analysis_id": "global_response_landscape_similarity",
        "file_prefix": "ResponseGlobal",
        "df": response_global,
        "x_col": "nearest_training_cell_response",
        "x_title": "Flattened DGE-landscape Pearson correlation",
        "labels": {name: ["SKM1", "U937"] for name, _ in OUTCOMES},
    },
    {
        "analysis_id": "drug_training_exposure",
        "file_prefix": "DrugExposure",
        "df": drug_exposure,
        "x_col": "drug_training_exposure_fraction",
        "x_title": "Drug training exposure fraction",
        "labels": {name: ["SKM1", "U937"] for name, _ in OUTCOMES},
    },
    {
        "analysis_id": "condition_training_exposure",
        "file_prefix": "ConditionExposure",
        "df": condition_exposure,
        "x_col": "condition_training_exposure_fraction",
        "x_title": "Drug–dose–duration training exposure fraction",
        "labels": {name: ["SKM1", "U937"] for name, _ in OUTCOMES},
    },
]

OUTCOME_CODES = {
    "delta_pcc": "DeltaPCC",
    "delta_r2": "DeltaR2",
    "dge_mse": "DGEMSE",
    "mse_gain_over_naive": "DGEMSEGain",
}

## 5. Generate all 20 standalone vector panels

In [ ]:
panel_exports = []
manifest_rows = []

for group in ANALYSIS_GROUPS:
    for outcome_index, (y_col, y_label) in enumerate(OUTCOMES, start=1):
        filename = (
            f"{group['file_prefix']}_{outcome_index}_"
            f"{OUTCOME_CODES[y_col]}_NoXTitle.pdf"
        )

        panel_data = plot_single_association_component(
            df=group["df"],
            analysis_id=group["analysis_id"],
            x_col=group["x_col"],
            y_col=y_col,
            y_label=y_label,
            selected_labels=group["labels"][y_col],
            filename=filename,
        )
        panel_exports.append(panel_data)

        manifest_rows.append({
            "analysis_id": group["analysis_id"],
            "panel_position_within_group": outcome_index,
            "outcome": y_col,
            "outcome_label": y_label,
            "x_column": group["x_col"],
            "x_axis_title_component": (
                f"XTitle_{group['file_prefix']}.pdf"
            ),
            "panel_pdf": filename,
            "selected_cell_labels": ",".join(group["labels"][y_col]),
            "panel_width_inches": SIZE_PANEL[0],
            "panel_height_inches": SIZE_PANEL[1],
        })

all_plot_data = pd.concat(panel_exports, ignore_index=True)
panel_manifest = pd.DataFrame(manifest_rows)

all_plot_data.to_csv(
    TABLE_OUT_DIR / "All_Analysis1A_Association_Component_PlotData.csv",
    index=False,
)
panel_manifest.to_csv(
    TABLE_OUT_DIR / "All_Analysis1A_Association_Component_Manifest.csv",
    index=False,
)

display(panel_manifest)

## 6. Export one fold legend and five standalone x-axis title components

In [ ]:
fold_handles = [
    Line2D(
        [0],
        [0],
        marker="o",
        linestyle="none",
        markerfacecolor=FOLD_COLORS[f"cell_split{i}"],
        markeredgecolor="none",
        markersize=6,
    )
    for i in range(1, 6)
]

save_standalone_legend(
    fold_handles,
    [f"Split {i}" for i in range(1, 6)],
    "Legend_All_Analysis1A_Folds_Above.pdf",
    ncol=5,
    figsize=(4.9, 0.55),
    fontsize=8.2,
)

for group in ANALYSIS_GROUPS:
    save_standalone_axis_title(
        group["x_title"],
        f"XTitle_{group['file_prefix']}.pdf",
        figsize=(4.9, 0.42),
    )

## 7. Output audit

In [ ]:
expected_panel_files = [
    OUT_DIR / row["panel_pdf"]
    for _, row in panel_manifest.iterrows()
]
expected_support_files = [
    OUT_DIR / "Legend_All_Analysis1A_Folds_Above.pdf",
    *[
        OUT_DIR / f"XTitle_{group['file_prefix']}.pdf"
        for group in ANALYSIS_GROUPS
    ],
]

missing = [
    str(path)
    for path in expected_panel_files + expected_support_files
    if not path.exists()
]
require(not missing, f"Expected output files are missing: {missing}")

print(f"Generated {len(expected_panel_files)} standalone association panels.")
print(f"Generated {len(expected_support_files)} support components.")
print("All files are located in:")
print(OUT_DIR)